# Introduction to Boto3

## What you will learn in this course 🧐🧐

Once you have an S3 bucket and zone folders such as `raw/`, `staging/`, and `curated/` in the console, you can repeat the same work from Python. This notebook shows how, so your FinPulse pipeline does not rely on clicking through the AWS console for every upload.

By the end of this lecture, you can:

- Connect to AWS from Python with Boto3
- Understand how Boto3 finds your credentials, and how to keep them safe
- Upload, list, and download objects in S3 using the `raw/crypto/` prefix pattern
- Catch a `ClientError` and read what it means

## What Is Boto3?

Boto3 is the official AWS SDK for Python. It wraps AWS APIs into Python functions, so you can build automation and data pipelines without writing raw HTTP requests.


<Note type="tip" title="What is an SDK?">

SDK stands for Software Development Kit. It is a set of libraries and tools that make it easier to use a service from code.

</Note>


In [1]:
# Install boto3
#
# Prefer adding this requirement to your project's requirements.txt.
!pip install "boto3>=1.34"

  Using cached s3transfer-0.19.2-py3-none-any.whl.metadata (1.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 41.1 MB/s  0:00:0040.1 MB/s eta 0:00:01
Using cached s3transfer-0.19.2-py3-none-any.whl (90 kB)
  Attempting uninstall: botocore
    Found existing installation: botocore 1.33.13
    Uninstalling botocore-1.33.13:
      Successfully uninstalled botocore-1.33.13
  Attempting uninstall: s3transfer━━━━━━━━━━━━━━━━━━━━ 0/3 [botocore]
    Found existing installation: s3transfer 0.8.2━ 0/3 [botocore]
    Uninstalling s3transfer-0.8.2:━━━━━━━━━━━━━━━━ 0/3 [botocore]
      Successfully uninstalled s3transfer-0.8.2━━━ 0/3 [botocore]
  Attempting uninstall: boto3━━━━━━━━━━━━━━━━━━━━━ 0/3 [botocore]
    Found existing installation: boto3 1.33.13━━━━ 0/3 [botocore]
    Uninstalling boto3-1.33.13:━━━━━━━━━━━━━━━━━━━ 0/3 [botocore]
      Successfully uninstalled boto3-1.33.13━━━━━━ 0/3 [botocore]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [boto3]


If the output shows 1.34 or newer, you are good to go.

For FinPulse S3 upload labs, the expected baseline is `boto3>=1.34`. If your environment shows conflicts with `aiobotocore`, create a fresh virtual environment and reinstall.


Official guides and API listings for this lecture are grouped under Resources at the end of this notebook.

## How Boto3 Finds Your Credentials

Boto3 searches for credentials in a fixed order. The first match wins.

```mermaid
flowchart TD
  A[Environment variables\nAWS_ACCESS_KEY_ID\nAWS_SECRET_ACCESS_KEY] --> B[~/.aws/credentials]
  B --> C[~/.aws/config]
  C --> D[EC2 instance metadata\nIAM role]
  D --> E[Boto3 uses the first match]
```


<Note type="warning">

Do not hardcode your access key ID or secret access key in a notebook cell. Use environment variables or `~/.aws/credentials`. If you accidentally commit a key to git, rotate it immediately in the IAM console.

</Note>


In [1]:
from dotenv import load_dotenv

In [2]:
load_dotenv()

False

In [3]:
import os
cle_amazon_1 = os.getenv("AWS_ACCESS_KEY_ID")
cle_amazon_1

In [6]:
%pwd

'/Users/thierrylegros/00-JEDHA/Fullstack-AI/M03-Data-Infrastructure/D02_data_collection/exos'

In [7]:
import boto3

# This uses the credentials resolution chain shown above.
# Configure credentials using environment variables or ~/.aws/credentials.
session = boto3.Session()

s3 = session.client("s3")

response = s3.list_buckets()
[b["Name"] for b in response.get("Buckets", [])]

NoCredentialsError: Unable to locate credentials

If the cell above returns your bucket names, your credentials are working.

If you lost your access keys, regenerate them in the IAM console. The short video below shows where to do that.

If you prefer environment variables, set `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` in your shell before starting Jupyter.


<Video video="https://vimeo.com/1128158131" />

## Uploading and Downloading Files

For FinPulse crypto data, the raw zone prefix is `raw/crypto/`.

The next cells upload a local file to that prefix, list the objects under the prefix, and then download it back to your machine.


In [8]:
from pathlib import Path
import json

bucket_name = "bucket-test-aifs-02"  # for example: finpulse-data-lake-yourname
s3_key = "raw/crypto/crypto_data.json"

local_path = Path("crypto_data.json")

if not local_path.exists():
    # Stand-in sample so the notebook stays runnable.
    local_path.write_text(
        json.dumps({"BTC": [{"date": "2021-01-01", "open": 29000, "close": 29300}]}),
        encoding="utf-8",
    )

str(local_path)

'crypto_data.json'

Upload the file into the raw zone. The third argument is the S3 object key, it is the full path inside your bucket.


In [9]:
from botocore.exceptions import ClientError

try:
    s3.upload_file(str(local_path), bucket_name, s3_key)

    objects = s3.list_objects_v2(Bucket=bucket_name, Prefix="raw/crypto/")
    [obj["Key"] for obj in objects.get("Contents", [])]
except ClientError as e:
    e.response.get("Error", {})

If the upload worked, you should see `raw/crypto/crypto_data.json` in the returned list.

If you see an error dict instead, the `Code` field usually tells you what to fix.


<Note type="tip">

A missing key usually means a typo in the string, a missing bucket usually means a wrong `bucket_name`, and `AccessDenied` means the IAM user does not have the required permission for this action.

</Note>


Download the same object back to your machine, then open it with `json.load` to verify the round trip.


In [5]:
download_path = Path("crypto_data_downloaded.json")

s3.download_file(bucket_name, s3_key, str(download_path))

downloaded = json.loads(download_path.read_text(encoding="utf-8"))
downloaded

{'BTC': [{'date': '2021-01-01', 'open': 29000, 'close': 29300}]}

In [12]:
missing_key = "raw/crypto/this-file-does-not-exist.json"

try:
    s3.download_file(bucket_name, missing_key, "does_not_matter.json")
except ClientError as e:
    e.response.get("Error", {})

In [11]:
error = None

try:
    s3.head_object(Bucket=bucket_name, Key=missing_key)
except ClientError as e:
    error = e.response.get("Error", {})

error_code = error.get("Code")
error_message = error.get("Message")

{"Code": error_code, "Message": error_message}

{'Code': '404', 'Message': 'Not Found'}

Boto3 exposes many S3 operations beyond upload and download. When you need something new, open the Boto3 S3 service reference from Resources at the end of this notebook and search for the method name.


## Client vs Resource


In the documentation, you will see both `client` and `resource`.

- `client` is the low level API. It exposes every S3 operation.
- `resource` is a higher level wrapper. It can feel nicer, but it does not cover everything.

For a data pipeline, `client` is a safe default.

If you want more detail on how `client`, `resource`, and `Session` differ, use the Stack Overflow thread linked under Resources at the end of this notebook.


## The Production Pattern

When you run Boto3 on your laptop, it reads credentials from environment variables or `~/.aws/credentials`.

In production on AWS, the usual pattern is different. You attach an IAM role to your EC2 instance (or to your Lambda function or ECS task). Boto3 then picks up short lived credentials automatically through instance metadata.


<Note type="important">

Use access keys for local development. Use IAM roles for code running on AWS. That removes the need to store long lived secrets anywhere.

</Note>


## Resources 📚📚

- [Boto3 quickstart](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/quickstart.html)
- [Boto3 credentials configuration](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/credentials.html)
- [Boto3 S3 service reference](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3.html)
- [Difference in boto3 between resource, client, and session](https://stackoverflow.com/questions/42809096/difference-in-boto3-between-resource-client-and-session)
